# BACCO responses in the numbered joint DES example

The numbered TATT YAML supplies CAMB and an absolute gravity-only matter
bispectrum. Enable its commented `bacco_bispectrum` theory block to supply the
paired ratios $R_P=P_{\rm hydro}/P_{\rm DMO}$ and
$R_B=B_{\rm hydro}/B_{\rm DMO}$. Both are needed for a joint two-/three-point
forecast. BACCO does not replace the absolute bispectrum theory block.

This notebook performs the edits directly on the ordinary Cobaya input. It
uses the shared Fisher functions; it does not launch an alternative prediction
pipeline. The five varied coordinates are $\sigma_8$, two feedback coordinates,
$A_1$ and $A_2$. All other cosmological and nuisance parameters are fixed, so
these are **conditional** errors. The simulation covariance remains fixed.
Quadratic IA retains the declared gravity-only treatment.

Numerical source edits have not been followed by a physical rerun; outputs are
empty. The cells require the installed BACCO emulator and its downloaded
weights. A laptop need not run this notebook merely to read its definitions.

These examples use the unmasked, uncompressed row order of numbered examples 1
and 2. A different dataset mask or compression requires matching row labels
before using the two-/three-point split shown here.

The companion [teaching note](baryonic_feedback.pdf) derives the power and
bispectrum response conversions and explains each comparison below.

The root [paired-response example](../../EXAMPLE_JOINT_FEEDBACK.ipynb) teaches
the interface with artificial tables; the
[feedback-scenario example](../../EXAMPLE_FEEDBACK_SCENARIOS.ipynb) compares
physical response models. Those demonstrations do not supply this notebook's
five-parameter forecast.


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import os
# BLAS must not start another worker team inside the native OpenMP calculation.
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"

from copy import deepcopy
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
from scipy.linalg import solve_triangular
from cobaya.model import get_model
from cobaya.yaml import yaml_load_file, yaml_dump_file

project = Path.cwd().resolve()
while not (project / "EXAMPLE_EVALUATE1.yaml").exists():
    if project == project.parent:
        raise FileNotFoundError("Open this notebook from the DES project directory.")
    project = project.parent
cocoa = project.parents[1]
sys.path.insert(0, str(project / "interface"))
sys.path.insert(0, str(cocoa / "external_modules/code/cosmolike_core"))
import cosmolike_des_3pcf_interface as ci
import cosmolike_notebook_utils.fisher as cnu

# Numbered Cocoa YAMLs resolve package and dataset paths from the Cocoa directory.
os.chdir(cocoa)

In [ ]:
# Figure style of the whole notebook: the shared plotting functions never change
# rcParams, so these lines set every font, tick and grid default.
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'

## 1. Enable the additional theory block

BACCO's ratios describe cold matter (cold dark matter plus baryons), whereas
lensing responds to total matter including massive neutrinos. The default
continuation combines the cold response with coherent linear neutrino fields.
Applying cold ratios directly to total matter is a different approximation;
the comparison below changes that choice while holding the fiducial fixed.

Triangles outside the emulator's wavenumber domain also need an explicit
continuation. Rescaling all sides preserves triangle shape. Clamping each side
separately generally changes that shape. Neither is a calibration of the
untrained region. Changes in the continuation are physical-model comparisons;
changes in table sampling are numerical checks.

In [ ]:
info = yaml_load_file(str(project / "EXAMPLE_EVALUATE2.yaml"))
fiducial = dict(info["sampler"]["evaluate"]["override"])
info.pop("sampler")
info.pop("output", None)
info["likelihood"]["des_3pcf.joint_shear"]["omp_threads"] = 4

info["likelihood"]["des_3pcf.joint_shear"]["external_baryon_response"] = True
# Keep the numbered YAML's absolute-bispectrum provider and append BACCO.
info["theory"]["bacco_bispectrum"] = {
    "response_settings": {
        "neutrino_model": "coherent_linear_neutrinos",
        "low_k_anchor": 0.02,
        "high_k_anchor": 10.0,
        "high_k_mode": "shape_preserving",
        "high_z_mode": "hold",
        "nz": 17,
        "nk": 97,
        "nu": 33,
        "nv": 9,
        "boost": 2,
    },
    "zmax": 3.1,
    "kmax_boltzmann": 5.0,
}
# The block accepts logarithmic feedback coordinates. Only the first two
# vary in this conditional forecast; the other three remain explicitly fixed.
info["params"]["M_c_baccoemu"] = {
    "prior": {"min": 10.02, "max": 15.99},
    "ref": 14.0,
}
info["params"]["eta_baccoemu"] = {
    "prior": {"min": -0.69, "max": 0.19},
    "ref": -0.3,
}
info["params"]["beta_baccoemu"] = -0.22
info["params"]["M1_z0_cen_baccoemu"] = 10.5
info["params"]["theta_inn_baccoemu"] = -0.86
fiducial["M_c_baccoemu"] = 14.0
fiducial["eta_baccoemu"] = -0.3
steps = {
    "sigma8": 0.005,
    "M_c_baccoemu": 0.02,
    "eta_baccoemu": 0.01,
    "a1": 0.01,
    "a2": 0.02,
}
# No Gaussian prior is added to these five varied coordinates.
priors = {}

In [ ]:
def get_dv(parameters):
    """Evaluate the current numbered Cobaya model at a physical parameter point.

    Cobaya runs its declared theory blocks before the DES likelihood. The
    likelihood retains the full mean it evaluated; the native selector applies
    its data mask and compression. A Fisher displacement must reach that same
    prediction, not another CAMB or bispectrum preparation route.

    Arguments:
      parameters = dictionary of sampled physical parameters for the current model.
    Returns:
      Owned one-dimensional selected data vector, in the covariance's row order.
    Raises:
      RuntimeError if the likelihood evaluation is not finite.
    Side effects:
      Updates the current model and the process-global native cosmology.
    """
    value = model.loglike(
        params_values=parameters,
        return_derived=False,
        cached=False,
    )
    if not np.isfinite(value):
        raise RuntimeError("A Fisher point has nonfinite likelihood; inspect that point.")
    return np.array(
        object=ci.select_joint_data_vector(prediction=likelihood.last_datavector),
        dtype=np.float64,
        copy=True,
    )

## 2. Separate model choices from numerical refinements

Each case builds and closes one Cobaya model. Only the named response settings
change; source distributions, cuts, priors and covariance remain fixed.
Refining an axis doubles its intervals, retaining the old nodes. Increasing the
dense interpolation factor tests the linear reader independently of those nodes.

The baseline and the three continuation tests have full derivatives at $h$ and
$h/2$. The response-grid cases check their means; the wavenumber refinement also
checks derivatives. These explicit choices keep the cost visible. Refine other
columns or axes if their mean changes suggest that their derivatives matter.

In [ ]:
# Each entry holds response settings, whether to compute derivatives, and
# whether the comparison changes the physical continuation or only numerics.
cases = {
    "baseline": ({}, True, "reference"),
    "cold_on_total": ({"neutrino_model": "cold_on_total"}, True, "model"),
    "side_clamp": ({"high_k_mode": "side_clamp"}, True, "model"),
    "high_k_5": ({"high_k_anchor": 5.0}, True, "model"),
    "low_k_004": ({"low_k_anchor": 0.04}, False, "model"),
    "high_z_fade": ({"high_z_mode": "fade_to_unity"}, False, "model"),
    "feedback_z": ({"nz": 33}, False, "numerics"),
    "feedback_k": ({"nk": 193}, True, "numerics"),
    "feedback_u": ({"nu": 65}, False, "numerics"),
    "feedback_v": ({"nv": 17}, False, "numerics"),
    "feedback_dense": ({"boost": 3}, False, "numerics"),
}
results = {}
for label, (settings, derivatives_needed, interpretation) in cases.items():
    case_info = deepcopy(info)
    case_info["theory"]["bacco_bispectrum"]["response_settings"].update(settings)
    model = get_model(info_or_yaml_or_file=case_info)
    likelihood = model.likelihood["des_3pcf.joint_shear"]
    try:
        covariance = np.array(object=ci.get_joint_covariance(), copy=True)
        prediction = cnu.ExactMeanCache(predict=get_dv)
        result = {
            "mean": prediction(parameters=fiducial),
            "covariance": covariance,
            "row_kind": likelihood.rows.kind.copy(),
            "row_bins": likelihood.rows.bins.copy(),
            "row_angles": likelihood.rows.angles.copy(),
        }
        if derivatives_needed:
            names, center, dh, dh2 = cnu.differentiate(
                predict=prediction,
                parameters=fiducial,
                steps=steps,
            )
            result["names"] = np.asarray(names)
            result["derivatives_h"] = dh
            result["derivatives_h2"] = dh2
            for step_label, derivative in (("h", dh), ("h2", dh2)):
                information = cnu.compare_information(
                    derivatives=derivative,
                    names=names,
                    kind=likelihood.rows.kind,
                    covariance=covariance,
                    priors=priors,
                )
                errors, status = cnu.marginalized_errors(
                    comparison=information,
                    names=names,
                )
                print(label, step_label, status)
                result[step_label + "_fisher_joint"] = information["fisher_joint"]
                result[step_label + "_fisher_two_point"] = information["fisher_two_point"]
                # Unresolved cases retain their Fisher matrices, not invented errors.
                if "sigma_joint" in errors:
                    result[step_label + "_sigma_joint"] = errors["sigma_joint"]
                if "sigma_two_point" in errors:
                    result[step_label + "_sigma_two_point"] = errors["sigma_two_point"]
        results[label] = result
        directory = project / "chains/fisher_bacco"
        directory.mkdir(parents=True, exist_ok=True)
        yaml_dump_file(
            file_name=str(directory / (label + ".yaml")),
            data=case_info,
            error_if_exists=False,
        )
        np.savez(file=directory / (label + ".npz"), **result)
    finally:
        model.close()
baseline = results["baseline"]

## 3. Compare the means using the complete covariance

For $\Delta\mu=\mu_{\rm case}-\mu_{\rm baseline}$, the joint distance is
$\Delta\mu^T C^{-1}\Delta\mu$. The two-point part uses the marginal covariance
$C_{22}$, not a submatrix of $C^{-1}$. The additional three-point contribution
uses the conditional covariance

$$S=C_{33}-C_{32}C_{22}^{-1}C_{23}$$

and the residual mean change

$$r=\Delta\mu_3-C_{32}C_{22}^{-1}\Delta\mu_2.$$

Thus the joint distance equals the two-point distance plus $r^TS^{-1}r$.
The shared routine performs these matrix operations and retains the cross block.
A large distance diagnoses sensitivity; it is not a posterior bias after fitting.

In [ ]:
for label, result in results.items():
    delta = result["mean"] - baseline["mean"]
    distance = cnu.compare_information(
        derivatives=delta[:, None],
        names=["model_difference"],
        kind=baseline["row_kind"],
        covariance=baseline["covariance"],
    )
    print(label, cases[label][2], "joint / xi / additional Map3 chi2:")
    print(
        distance["fisher_joint_likelihood"][0, 0],
        distance["fisher_two_point_likelihood"][0, 0],
        distance["fisher_conditional_three_point"][0, 0],
    )

## 4. Inspect feedback and IA derivatives

The comparison uses declared *display* displacements, $\Delta\log_{10}M_c=0.2$
and $\Delta\log_{10}\eta=0.1$, distinct from the smaller derivative steps.
Normalize each row by its standard deviation. Fitting one response to another
with the full covariance shows which part cannot be imitated by that parameter.
A five-parameter marginalized forecast below answers the broader question.

In [ ]:
response_plots = importlib.import_module(
    name="cosmolike_notebook_utils.3pcf.derivative_templates",
)
names = baseline["names"].tolist()
derivatives = baseline["derivatives_h2"]
shapes = response_plots.compare_templates(
    target=0.2 * derivatives[:, names.index("M_c_baccoemu")],
    mimic=0.1 * derivatives[:, names.index("eta_baccoemu")],
    covariance=baseline["covariance"],
    two_point=baseline["row_kind"] != "map3",
)
for probe in ("two_point", "joint"):
    print(probe, "response cosine:", shapes[probe]["cosine"])
panels = []
for kind in ("xip", "xim", "map3"):
    bins = baseline["row_bins"]
    selected = (baseline["row_kind"] == kind) & (bins[:, 0] == 3) & (bins[:, 1] == 3)
    angle_column = 2
    if kind == "map3":
        selected &= bins[:, 2] == 3
        angle_column = 0
    indices = np.flatnonzero(selected)
    panels.append({
        "coordinate": baseline["row_angles"][indices, angle_column] * 10800.0 / np.pi,
        "templates": shapes["templates"][:, indices],
        "residuals": np.array([
            shapes["two_point"]["residual"][indices],
            shapes["joint"]["residual"][indices],
        ]),
        "title": {"xip": r"$\xi_+$", "xim": r"$\xi_-$",
                      "map3": r"$M_{\mathrm{ap}}^3$"}[kind] + ", source bin 4",
        "coordinate_label": "Angle [arcmin]",
    })
response_plots.plot_derivative_templates(
    panels=panels,
    parameter_labels=[r"$\Delta\log_{10}M_c=0.2$", r"$\Delta\log_{10}\eta=0.1$"],
    show=1,
)

## 5. Quantify the information gain and numerical sensitivity

Compare joint and two-point errors only when both parameter matrices are
resolved. A pseudoinverse would hide an unconstrained combination. Invert the
entire five-parameter Fisher matrix before extracting the feedback pair; this
marginalizes the amplitude and IA coordinates. These comparisons hold the
other cosmological and source parameters fixed.

A stable fiducial mean does not establish derivative convergence. Compare both
step sizes and the independently refined wavenumber grid. Physical continuation
changes must remain separate from numerical refinement in the interpretation.

In [ ]:
if "h2_sigma_joint" in baseline and "h2_sigma_two_point" in baseline:
    for index, name in enumerate(names):
        ratio = baseline["h2_sigma_joint"][index] / baseline["h2_sigma_two_point"][index]
        print(name, "joint / xi error:", ratio)
for label in ("baseline", "cold_on_total", "side_clamp", "high_k_5", "feedback_k"):
    result = results[label]
    if "h_sigma_joint" not in result or "h2_sigma_joint" not in result:
        print(label, "unresolved parameter direction; no marginalized FoM")
        continue
    ratio = result["h2_sigma_joint"] / result["h_sigma_joint"]
    print(label, "h/2 error divided by h error:", ratio)
    for probe in ("two_point", "joint"):
        if "h2_sigma_" + probe not in result:
            continue
        fom = cnu.figure_of_merit(
            fisher=result["h2_fisher_" + probe],
            names=names,
            pair=("M_c_baccoemu", "eta_baccoemu"),
        )
        print(label, probe, "feedback-pair FoM:", fom)

### Check whether the local feedback interval reaches the declared bounds

A Fisher error is a local Gaussian width, not an interval clipped by a flat
prior. Compare the fiducial plus or minus one marginalized error with the
feedback bounds explicitly declared above. These are the notebook's parameter
bounds; checking them does not independently verify every emulator training
coordinate. If the interval crosses a bound, a bounded posterior calculation
is needed before interpreting it as an allowed parameter interval.


In [ ]:
# The same parameter bounds used to build Cobaya define this check. No
# additional emulator is constructed and no physical prediction is evaluated.
if "h2_sigma_joint" in baseline:
    for name in ("M_c_baccoemu", "eta_baccoemu"):
        index = names.index(name)
        sigma = baseline["h2_sigma_joint"][index]
        lower = fiducial[name] - sigma
        upper = fiducial[name] + sigma
        bounds = info["params"][name]["prior"]
        print(name, "local one-sigma interval:", lower, upper)
        print("Declared bounds:", bounds["min"], bounds["max"])
        if lower < bounds["min"] or upper > bounds["max"]:
            print("The local interval crosses a declared bound; "
                  "it is not a bounded posterior interval.")


The emulator's training cosmologies and feedback ranges still constrain which
fiducial points can be used. The response table's continuation and neutrino
conversion are explicit assumptions, not emulator predictions outside its
training domain. Matching a published result also requires its source model,
angular cuts, finite-triangle averaging, covariance prescription and parameter
priors. The notebook demonstrates the supplied joint response and tests its
local consequences without claiming that agreement.